# 04 · Memory — an agent's evolving knowledge graph

Entities + relations + observations. Run `build.py` first to seed the `mcp_memory` graph. This notebook puts the seed back before it reads, so it shows the same memory however many times it, or `ask.py`, has run before.

In [1]:
import warnings; warnings.filterwarnings("ignore")   # quiet 3rd-party import warnings
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # examples/demos
from _common import clients, console
print("helpers ready — no API key needed (the MCP servers are pure tools)")

helpers ready — no API key needed (the MCP servers are pure tools)


## Read, search, and exact lookup

In [2]:
from _common import memory_seed

async def explore():
    async with clients.memory_client("mcp_memory","memory") as mem:
        # Start from the seed whatever the last run of ask.py left, so this reads the same
        # memory every time. create_entities merges by name and create_relations skips one
        # already there, so it is the same work on a memory that is already seeded.
        await memory_seed.restore(mem)
        g=clients.data(await mem.call_tool("read_graph",{}))
        console.kv("entities", sorted(e["name"] for e in g["entities"]))
        console.kv("truncated@limit2", clients.data(await mem.call_tool("read_graph",{"limit":2}))["truncated"])
        console.kv("search 'Seoul'", sorted(e["name"] for e in clients.data(await mem.call_tool("search_memories",{"query":"Seoul"}))["entities"]))
        found=clients.data(await mem.call_tool("find_memories_by_name",{"names":["Alex Kim"]}))
        console.kv("Alex Kim relations", [f"-{r['relationType']}->{r['target']}" for r in found["relations"]])
await explore()

  entities                   ['Alex Kim', 'Incheon International', 'Korean Air', 'Tokyo Haneda', 'Tokyo Trip 2026']
  truncated@limit2           True
  search 'Seoul'             ['Alex Kim', 'Incheon International']
  Alex Kim relations         ['-LIVES_NEAR->Incheon International', '-FLIES_WITH->Korean Air']


## Evolve: add then delete an observation

In [3]:
async def evolve():
    async with clients.memory_client("mcp_memory","memory") as mem:
        await mem.call_tool("add_observations",{"observations":[{"entityName":"Alex Kim","observations":["Speaks Korean and English"]}]})
        obs=lambda g: clients.data(g)["entities"][0]["observations"]
        console.kv("after add", obs(await mem.call_tool("find_memories_by_name",{"names":["Alex Kim"]})))
        await mem.call_tool("delete_observations",{"deletions":[{"entityName":"Alex Kim","observations":["Speaks Korean and English"]}]})
        console.kv("after delete", obs(await mem.call_tool("find_memories_by_name",{"names":["Alex Kim"]})))
await evolve()

  after add                  ['Frequent flyer', 'Based in Seoul', 'Prefers window seats', 'Speaks Korean and English']
  after delete               ['Frequent flyer', 'Based in Seoul', 'Prefers window seats']
